[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gnoejh/AIBookGitHub/blob/main/15_execution.ipynb)


# AI System Architecture: Hierarchy - Bottom Layer

## Overview: The AI System Hierarchy

AI systems can be understood through **two perspectives**:

1. **Hierarchy** (Vertical) - Layers from raw models to intelligent agents
2. **Networking** (Horizontal) - Communication protocols between components

This notebook covers **Layer 1: The Bottom Layer** - where AI models actually run.

### Complete Hierarchy (Overview)

```
┌─────────────────────────────────────────┐
│  Layer 4: AGENTS & ORCHESTRATION        │  ← Multi-agent systems, workflows
├─────────────────────────────────────────┤
│  Layer 3: STATE MANAGEMENT              │  ← Conversation history, context
├─────────────────────────────────────────┤
│  Layer 2: API CLIENTS & ABSTRACTION     │  ← HTTP clients, LiteLLM, SDKs
├─────────────────────────────────────────┤
│  Layer 1: MODEL PROVIDERS (THIS LAYER)  │  ← Where models actually execute
└─────────────────────────────────────────┘
```

---

## Layer 1: Model Provider Layer (Bottom)

This is where **AI models actually run** - the computational foundation of everything above it.

## 1.1 Understanding Model Providers

### What is a Model Provider?

A **model provider** is the entity that:
1. **Creates** the AI model (trains it on data)
2. **Hosts** the model infrastructure (servers/GPUs)
3. **Executes** inference (runs the model to generate responses)
4. **Exposes** an API for access

### Three Types of Model Providers

| Type | Examples | Ownership | Access Method |
|------|----------|-----------|---------------|
| **Proprietary Cloud** | OpenAI, Anthropic, Google | Closed-source, cloud-hosted | API only |
| **Open Model Cloud** | OpenRouter, Groq, Together AI, HuggingFace | Open models, cloud-hosted | API only |
| **Local Self-Hosted** | Ollama, LM Studio, vLLM | Open models, your hardware | Local API |

### Visual Representation

```
PROPRIETARY CLOUD PROVIDERS
┌──────────────────────────────────────────────────────┐
│  OpenAI Data Centers                                 │
│  ├─ GPT-4 (proprietary weights)                      │
│  ├─ Thousands of GPUs                                │
│  └─ API: https://api.openai.com/v1/chat/completions  │
└──────────────────────────────────────────────────────┘
                    ↕ HTTPS
              Your Application

OPEN MODEL CLOUD PROVIDERS
┌──────────────────────────────────────────────────────┐
│  Groq Data Centers                                   │
│  ├─ LLaMA 3.2 (open weights from Meta)              │
│  ├─ Custom LPU chips (ultra-fast)                   │
│  └─ API: https://api.groq.com/v1/chat/completions   │
└──────────────────────────────────────────────────────┘
                    ↕ HTTPS
              Your Application

LOCAL SELF-HOSTED
┌──────────────────────────────────────────────────────┐
│  Your Computer                                       │
│  ├─ Ollama Server (localhost:11434)                 │
│  ├─ LLaMA 3.2 (downloaded to disk)                  │
│  ├─ Your GPU/CPU                                    │
│  └─ API: http://localhost:11434/v1/chat/completions │
└──────────────────────────────────────────────────────┘
                    ↕ Local HTTP
              Your Application
```

## 1.2 Model Provider Characteristics

### Key Attributes Comparison

| Attribute | Proprietary Cloud | Open Model Cloud | Local Self-Hosted |
|-----------|-------------------|------------------|-------------------|
| **Model Weights** | Closed (secret) | Open (downloadable) | Open (you own copy) |
| **Where It Runs** | Their servers | Their servers | Your hardware |
| **Data Privacy** | Sent to cloud | Sent to cloud | Never leaves your machine |
| **Rate Limits** | Yes (per API key) | Yes (per API key) | No (unlimited) |
| **Internet Required** | Yes | Yes | No (after download) |
| **Cost** | Per-token pricing | Per-token or free tier | Hardware cost only |
| **Speed** | Fast | Very fast (Groq) | Depends on your GPU |
| **Model Quality** | Highest (GPT-4) | Good (LLaMA 3.2) | Good (same open models) |
| **Uptime** | 99.9% SLA | Variable | Depends on you |
| **Customization** | Fine-tune only | Fine-tune only | Full control |

### Critical Insight

**The model provider layer is stateless** - it doesn't remember previous requests. Each API call is independent:

```python
# Call 1
response1 = provider.complete("My name is Alice")

# Call 2 - provider has NO memory of Call 1
response2 = provider.complete("What's my name?")  
# ❌ Model doesn't know you're Alice
```

**State management happens in Layer 3** (covered in next notebook).

## 1.3 Code Examples: Direct Provider Access

### Example 1: Proprietary Cloud Provider (OpenAI)

In [1]:
# Install required package
# !pip install openai python-dotenv

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from pathlib import Path

# Load .env from project root
# In Jupyter notebooks, go up from current directory to find project root
project_root = Path('.').resolve()
# Go up maximum 3 levels (from Agents/Stacks/ to project root)
for _ in range(3):
    if (project_root / 'pyproject.toml').exists():
        break
    if project_root.parent == project_root:  # Reached filesystem root
        break
    project_root = project_root.parent

env_path = project_root / '.env'
load_dotenv(dotenv_path=env_path)

# Direct connection to OpenAI's model provider
client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY')  # Your API key
)

# Call the provider - model executes on OpenAI's servers
response = client.chat.completions.create(
    model="gpt-3.5-turbo",  # Free tier model (similar size to LLaMA 3.1 8B for consistency)
    messages=[
        {"role": "user", "content": "Explain what a model provider is in one sentence."}
    ]
)

print(f"Provider: OpenAI (Proprietary Cloud)")
print(f"Model: gpt-3.5-turbo (Free tier - for consistency with other examples)")
print(f"Response: {response.choices[0].message.content}")
print(f"\nMetadata:")
print(f"  Tokens used: {response.usage.total_tokens}")
print(f"  Model version: {response.model}")

Provider: OpenAI (Proprietary Cloud)
Model: gpt-3.5-turbo (Free tier - for consistency with other examples)
Response: A model provider is a company or platform that offers access to a range of pre-trained machine learning models for use in various applications.

Metadata:
  Tokens used: 44
  Model version: gpt-3.5-turbo-0125


**What happened:**
1. Your code sent an HTTPS request to `https://api.openai.com/v1/chat/completions`
2. OpenAI's servers loaded GPT-4o-mini into memory (on their GPUs)
3. Model processed your input and generated output
4. Response sent back to you
5. Model unloaded from memory (stateless)

**The model never touched your computer** - it ran entirely on OpenAI's infrastructure.

### Example 2: Open Model Cloud Provider (Groq)

In [3]:
from openai import OpenAI  # Groq uses OpenAI-compatible API
import os
from dotenv import load_dotenv
from pathlib import Path

# Load .env from project root
project_root = Path('.').resolve()
# Go up maximum 3 levels (from Agents/Stacks/ to project root)
for _ in range(3):
    if (project_root / 'pyproject.toml').exists():
        break
    if project_root.parent == project_root:  # Reached filesystem root
        break
    project_root = project_root.parent

env_path = project_root / '.env'
load_dotenv(dotenv_path=env_path)

# Try GROQ_API_KEY first, fallback to GROQ_API_KEY1
groq_api_key = os.getenv('GROQ_API_KEY') or os.getenv('GROQ_API_KEY1')

# Direct connection to Groq's model provider
client = OpenAI(
    api_key=groq_api_key,
    base_url='https://api.groq.com/openai/v1'  # Point to Groq servers
)

# Call the provider - model executes on Groq's LPU chips
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",  # LLaMA 3.1 8B - free, fast model (consistent across examples)
    messages=[
        {"role": "user", "content": "Explain what a model provider is in one sentence."}
    ]
)

print(f"Provider: Groq (Open Model Cloud)")
print(f"Model: llama-3.1-8b-instant (LLaMA 3.1 8B - Free & Fast)")
print(f"Response: {response.choices[0].message.content}")
print(f"\nMetadata:")
print(f"  Tokens used: {response.usage.total_tokens}")
print(f"  Provider: Groq (ultra-fast LPU inference)")

Provider: Groq (Open Model Cloud)
Model: llama-3.1-8b-instant (LLaMA 3.1 8B - Free & Fast)
Response: A model provider is an individual or organization that offers or distributes pre-trained or custom models, often machine learning or artificial intelligence-based, to users or developers who can integrate and utilize these models within their projects or applications.

Metadata:
  Tokens used: 90
  Provider: Groq (ultra-fast LPU inference)


**What happened:**
1. Your code sent an HTTPS request to `https://api.groq.com/openai/v1/chat/completions`
2. Groq's servers loaded LLaMA 3.1 into their custom LPU chips
3. Model processed your input (very fast - Groq's specialty)
4. Response sent back to you

**Key difference from OpenAI:**
- Model weights are **open source** (Meta released them)
- Groq just hosts and runs them on specialized hardware
- You could download the same model and run it locally

### Example 3: Local Self-Hosted Provider (Ollama)

In [4]:
# First, install Ollama and pull a model:
# 1. Download from https://ollama.ai
# 2. Run: ollama pull llama3.2
# 3. Ollama server starts automatically at localhost:11434

In [5]:
from openai import OpenAI  # Ollama also uses OpenAI-compatible API
import requests

# Connect to LOCAL model provider
client = OpenAI(
    api_key='ollama',  # Not actually used, but required by API
    base_url='http://localhost:11434/v1'  # Local server!
)

# First, check if Ollama is running and what models are available
try:
    # Check if Ollama is running
    response = requests.get('http://localhost:11434/api/tags', timeout=2)
    if response.status_code == 200:
        models = response.json().get('models', [])
        model_names = [m.get('name', '') for m in models]
        print(f"Available Ollama models: {model_names if model_names else 'None'}")
        
        # Prefer LLaMA 3.1 8B for consistency with other examples, then other small free models
        preferred_models = [
            "llama3.1",       # LLaMA 3.1 8B - matches Groq example (consistent across examples)
            "llama3.2",       # LLaMA 3.2 (if 3.1 not available)
            "phi3",           # Microsoft's small model (~3.8GB) - very efficient
            "llama3.2:1b",   # Tiny 1B parameter version
            "gemma:2b",       # Google's small model
            "mistral",        # 7B model (if available)
        ]
        
        model_to_use = None
        for preferred in preferred_models:
            # Check if any installed model matches (handles tags like "llama3.2:latest")
            for installed_model in model_names:
                if preferred.split(':')[0] in installed_model:
                    model_to_use = installed_model.split(':')[0]  # Use base name
                    print(f"✅ Using model: {model_to_use}")
                    break
            if model_to_use:
                break
        
        # If no preferred model found, use first available
        if not model_to_use and model_names:
            model_to_use = model_names[0].split(':')[0]
            print(f"⚠️  Using first available model: {model_to_use}")
        elif not model_names:
            print("❌ No models installed.")
            print("   Install LLaMA 3.1 8B for consistency with other examples:")
            print("   - ollama pull llama3.1     (recommended, ~4.7GB, matches Groq example)")
            print("   Or other small free models:")
            print("   - ollama pull phi3          (~3.8GB)")
            print("   - ollama pull llama3.2:1b  (tiny, ~1.3GB)")
            print("   - ollama pull gemma:2b     (small, ~1.7GB)")
            raise Exception("No Ollama models available")
    else:
        raise Exception("Ollama server not responding")
except requests.exceptions.RequestException:
    print("❌ Ollama is not running or not installed.")
    print("   Install from: https://ollama.ai")
    print("   Then run: ollama pull llama3.1  (LLaMA 3.1 8B - matches other examples)")
    raise

# Call the provider - model executes on YOUR computer
try:
    response = client.chat.completions.create(
        model=model_to_use,  # Model downloaded to your disk
        messages=[
            {"role": "user", "content": "Explain what a model provider is in one sentence."}
        ]
    )
    
    print(f"\nProvider: Ollama (Local Self-Hosted)")
    print(f"Model: {model_to_use} (running on YOUR hardware)")
    print(f"Response: {response.choices[0].message.content}")
    print(f"\nMetadata:")
    print(f"  Location: Your computer (localhost)")
    print(f"  Privacy: Complete (no data sent to cloud)")
    print(f"  Cost: $0 (uses your GPU/CPU)")
except Exception as e:
    print(f"\n❌ Error calling Ollama: {e}")
    print("\nTo fix this:")
    print("1. Make sure Ollama is installed: https://ollama.ai")
    print("2. Install LLaMA 3.1 8B for consistency:")
    print("   - ollama pull llama3.1     (recommended, ~4.7GB, matches Groq example)")
    print("   Or other small free models:")
    print("   - ollama pull phi3         (~3.8GB, very efficient)")
    print("   - ollama pull llama3.2:1b  (tiny, ~1.3GB)")
    print("   - ollama pull gemma:2b    (small, ~1.7GB)")

Available Ollama models: ['llama3.2:latest']
✅ Using model: llama3.2

Provider: Ollama (Local Self-Hosted)
Model: llama3.2 (running on YOUR hardware)
Response: A model provider is a software component that supplies instances of data models, which are representations of data structures or business entities, to an application or system for use in data access, business logic, and other purposes.

Metadata:
  Location: Your computer (localhost)
  Privacy: Complete (no data sent to cloud)
  Cost: $0 (uses your GPU/CPU)


**What happened:**
1. Your code sent an HTTP request to `http://localhost:11434/v1/chat/completions`
2. Ollama server (running on your computer) loaded llama3.2 from disk
3. Model executed on YOUR GPU/CPU
4. Response returned to your code
5. **No internet connection used** (after initial model download)

**Critical difference:**
- Model weights stored on **your disk** (~2-4GB)
- Inference runs on **your hardware**
- **No API key needed** (you own the model)
- **Complete privacy** - no data sent anywhere

## 1.4 Understanding Provider APIs

### The Standard Interface

All three provider types expose similar APIs (OpenAI format is the de facto standard):

```python
response = client.chat.completions.create(
    model="<model_identifier>",  # Which model to use
    messages=[                   # Conversation history
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Hello"}
    ],
    temperature=0.7,            # Randomness (0-2)
    max_tokens=100,             # Max response length
    stream=False                # Streaming vs complete response
)
```

### Request Flow Diagram

```
Your Code                    Provider                    Model
    │                           │                          │
    │──── HTTP POST ───────────>│                          │
    │   {messages, model}       │                          │
    │                           │──── Load Model ────────>│
    │                           │                          │
    │                           │<─── Model Ready ────────│
    │                           │                          │
    │                           │──── Run Inference ─────>│
    │                           │                          │
    │                           │<─── Generated Text ─────│
    │                           │                          │
    │<──── HTTP Response ───────│                          │
    │   {choices, usage}        │                          │
    │                           │──── Unload Model ──────>│
    │                           │                          │
```

## 1.5 Comparing Provider Response Times

In [6]:
import time
from openai import OpenAI
import os

def benchmark_provider(name, client, model, prompt):
    """Measure response time for a provider"""
    start = time.time()
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50
        )
        elapsed = time.time() - start
        
        print(f"\n{'='*60}")
        print(f"Provider: {name}")
        print(f"Model: {model}")
        print(f"Time: {elapsed:.2f} seconds")
        print(f"Response: {response.choices[0].message.content[:100]}...")
        print(f"Tokens: {response.usage.total_tokens}")
        print(f"Tokens/sec: {response.usage.total_tokens / elapsed:.1f}")
        
    except Exception as e:
        print(f"\n{name}: Error - {str(e)}")

# Test prompt
prompt = "Explain neural networks in simple terms."

print("Benchmarking Model Providers...\n")

# 1. OpenAI (Proprietary Cloud)
if os.getenv('OPENAI_API_KEY'):
    openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    benchmark_provider("OpenAI (Proprietary)", openai_client, "gpt-4o-mini", prompt)

# 2. Groq (Open Model Cloud)
if os.getenv('GROQ_API_KEY'):
    groq_client = OpenAI(
        api_key=os.getenv('GROQ_API_KEY'),
        base_url='https://api.groq.com/openai/v1'
    )
    benchmark_provider("Groq (Open Cloud)", groq_client, "llama-3.1-8b-instant", prompt)

# 3. Ollama (Local)
try:
    ollama_client = OpenAI(
        api_key='ollama',
        base_url='http://localhost:11434/v1'
    )
    benchmark_provider("Ollama (Local)", ollama_client, "llama3.2", prompt)
except:
    print("\nOllama: Not available (install from ollama.ai)")

print(f"\n{'='*60}")
print("\nTypical Results:")
print("  Groq:    0.5-1s   (ultra-fast LPU chips)")
print("  OpenAI:  1-2s     (optimized cloud infrastructure)")
print("  Ollama:  2-10s    (depends on your GPU/CPU)")

Benchmarking Model Providers...


Provider: OpenAI (Proprietary)
Model: gpt-4o-mini
Time: 2.64 seconds
Response: Sure! Neural networks are a type of computer program designed to recognize patterns and make decisio...
Tokens: 64
Tokens/sec: 24.2

Provider: Groq (Open Cloud)
Model: llama-3.1-8b-instant
Time: 0.48 seconds
Response: **Neural Networks in Simple Terms**

Imagine your brain is made up of billions of tiny computers cal...
Tokens: 93
Tokens/sec: 193.0

Provider: Ollama (Local)
Model: llama3.2
Time: 9.92 seconds
Response: Neural networks are a type of computer system that mimics the way our brains work. They're made up o...
Tokens: 83
Tokens/sec: 8.4


Typical Results:
  Groq:    0.5-1s   (ultra-fast LPU chips)
  OpenAI:  1-2s     (optimized cloud infrastructure)
  Ollama:  2-10s    (depends on your GPU/CPU)


## 1.6 Provider Selection Decision Tree

### When to Choose Each Provider Type

```
START: Which provider should I use?
    │
    ├─ Need BEST quality model? ──> OpenAI/Anthropic (Proprietary Cloud)
    │                                 • GPT-4, Claude 3.5
    │                                 • Highest capability
    │                                 • Pay per token
    │
    ├─ Need FASTEST inference? ───> Groq (Open Model Cloud)
    │                                 • LLaMA, Mixtral
    │                                 • 10x faster than others
    │                                 • Free tier: 14,400 req/day
    │
    ├─ Need PRIVACY/Offline? ─────> Ollama (Local Self-Hosted)
    │                                 • Any open model
    │                                 • No data leaves your machine
    │                                 • Unlimited requests
    │
    └─ Need MANY models easily? ──> OpenRouter (Aggregator - covered in Layer 2)
                                     • 200+ models
                                     • One API key
                                     • But rate limited
```

### Cost Comparison for 1 Million Tokens

| Provider | Model | Input Cost | Output Cost | Total (50/50 mix) |
|----------|-------|------------|-------------|-------------------|
| **OpenAI** | GPT-4o-mini | $0.150 | $0.600 | **$0.375** |
| **Groq** | LLaMA 3.2 90B | $0.090 | $0.090 | **$0.090** |
| **Google** | Gemini Flash | $0.000 | $0.000 | **$0.000 (free)** |
| **Ollama** | LLaMA 3.2 | $0.000 | $0.000 | **$0.000 (hardware only)** |

*Free tiers have rate limits - see next notebook for details*

## 1.7 Key Takeaways - Bottom Layer

### Essential Concepts

1. **Model Provider = Where Models Actually Run**
   - Not just an API endpoint
   - Physical GPUs executing neural network computations
   - Can be cloud (OpenAI, Groq) or local (Ollama)

2. **All Providers Are Stateless**
   - Each request is independent
   - No memory between calls
   - You must manage conversation history (Layer 3)

3. **Three Provider Types**
   - **Proprietary Cloud**: Best quality, highest cost (OpenAI, Anthropic)
   - **Open Model Cloud**: Fast, moderate cost (Groq, Together AI)
   - **Local Self-Hosted**: Private, unlimited, hardware cost (Ollama)

4. **Same Interface, Different Backends**
   - OpenAI API format is the standard
   - Change `base_url` to switch providers
   - Same code works everywhere

5. **Trade-offs Are Fundamental**
   - Speed vs Quality vs Privacy vs Cost
   - No "best" provider - only best for your needs
   - Can mix multiple providers (covered in Layer 2)

### Next: Layer 2 - API Clients & Abstraction

In the next notebook, we'll cover:
- How to connect to providers (HTTP clients, SDKs)
- Aggregators like OpenRouter and LiteLLM
- Abstracting away provider differences
- Rate limiting and error handling

## Practice Exercises

### Exercise 1: Compare Three Providers

Send the same prompt to three different providers and compare:
- Response quality
- Speed
- Token usage

```python
prompt = "Explain the difference between AI and ML in 2 sentences."

# TODO: Call OpenAI, Groq, and Ollama with this prompt
# Compare the results
```

### Exercise 2: Measure Provider Latency

Create a function that measures time-to-first-token (TTFT) for streaming responses:

```python
def measure_ttft(client, model, prompt):
    # TODO: Implement streaming and measure time to first token
    pass
```

### Exercise 3: Calculate Costs

For a chatbot that processes 100,000 messages/day (avg 500 tokens each), calculate monthly costs for:
- OpenAI GPT-4o-mini
- Groq LLaMA 3.2
- Ollama (assume $2000 GPU purchase)

At what point does local hosting become cheaper?

---

**Next Notebook:** `2_hierarchy_api_layer.ipynb` - API Clients & Abstraction Layer